# 🚀 KHMER MASTER CRYPTO | APEX SUPER BRAIN AGI v13.00
## 🏛️ 10-Pillar Super Smart Model Training Suite (Google Colab GPU)

This notebook trains institutional-grade quantitative machine learning models based on the **10 Foundational Pillars**:
1. **Market Structure / Price Action (BOS, CHoCH, Swing HH/HL/LH/LL)**
2. **Support & Resistance / Supply & Demand (Order Blocks & Fair Value Gaps FVG)**
3. **Liquidity / Swings (PDH/PDL Turtle Soup Sweeps)**
4. **Multi-Timeframe EMA Alignment (EMA 20, 50, 200)**
5. **ATR Volatility & Dynamic Stop-Loss Normalization**
6. **Volume & Order Flow (CVD Imbalance & Taker Buy Delta)**
7. **VWAP & Standard Deviation Bands (±1σ, ±2σ)**
8. **RSI 14 & Bullish/Bearish Divergence Detection**
9. **MACD Momentum Acceleration**
10. **Fibonacci Retracement (Golden Pocket 61.8% OTE Pullback)**

### Models Trained:
- 🔮 **HMM Market Regime Classifier** (`brain_hmm_regime.pkl`)
- 🌳 **Triple Ensemble Confluence Classifier** (`brain_xgb.pkl`, `brain_catboost.pkl`, `brain_lightgbm.pkl`)
- 🧬 **PatchTST Self-Attention Transformer** (`brain_patchtst.h5`)
- 🔀 **Mixture-of-Experts (MoE) Gating Router** (`brain_moe_router.pkl`)

All trained model artifacts are automatically uploaded to **Hugging Face Model Hub** (`hemsinath/apex-ai-brain-models`), allowing the VPS and bot to sync instantly via `python sync_local_models.py`!

### 📦 Step 1: Install Quant & Machine Learning Dependencies

In [ ]:
!pip install --quiet xgboost catboost lightgbm hmmlearn huggingface_hub scikit-learn tensorflow requests pandas numpy
print("✅ Dependencies successfully installed!")

### 📊 Step 2: Binance Multi-Pair 15m/1h Kline Data Ingestion

In [ ]:
import os
import sys
import time
import json
import joblib
import requests
import numpy as np
import pandas as pd
from datetime import datetime

def fetch_binance_klines(symbol="BTCUSDT", interval="15m", limit=1000):
    url = f"https://api.binance.com/api/v3/klines?symbol={symbol}&interval={interval}&limit={limit}"
    res = requests.get(url, timeout=10)
    if res.status_code == 200:
        raw = res.json()
        cols = ['timestamp', 'open', 'high', 'low', 'close', 'volume',
                'close_time', 'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore']
        df = pd.DataFrame(raw, columns=cols)
        for c in ['open', 'high', 'low', 'close', 'volume', 'taker_buy_base', 'taker_buy_quote']:
            df[c] = pd.to_numeric(df[c])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        df.set_index('timestamp', inplace=True)
        return df
    return pd.DataFrame()

SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT"]
dfs = []
print(f"Fetching data for: {SYMBOLS}...")
for s in SYMBOLS:
    df_s = fetch_binance_klines(s, interval="15m", limit=1000)
    if not df_s.empty:
        df_s['symbol'] = s
        dfs.append(df_s)
    time.sleep(0.3)

raw_df = pd.concat(dfs).sort_index()
print(f"✅ Total candles collected: {len(raw_df)}")
raw_df.tail()

### 🧠 Step 3: Compute the 10 Foundational Quantitative Pillars

In [ ]:
def compute_10_pillar_features(df):
    df = df.copy()
    # Pillar 4: EMA Multi-Timeframe
    df['ema20'] = df['close'].ewm(span=20, adjust=False).mean()
    df['ema50'] = df['close'].ewm(span=50, adjust=False).mean()
    df['ema200'] = df['close'].ewm(span=min(200, len(df)), adjust=False).mean()
    df['ema_alignment'] = np.where((df['close'] > df['ema20']) & (df['ema20'] > df['ema50']) & (df['close'] > df['ema200']), 1.0,
                          np.where((df['close'] < df['ema20']) & (df['ema20'] < df['ema50']) & (df['close'] < df['ema200']), -1.0, 0.0))

    # Pillar 5: ATR Volatility
    high_low = df['high'] - df['low']
    high_close = (df['high'] - df['close'].shift()).abs()
    low_close = (df['low'] - df['close'].shift()).abs()
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    df['atr14'] = ranges.max(axis=1).rolling(14).mean()
    df['atr_normalized'] = df['atr14'] / (df['close'] + 1e-10)

    # Pillar 6: Volume & Order Flow (CVD Imbalance)
    df['taker_sell_base'] = df['volume'] - df['taker_buy_base']
    df['delta_volume'] = df['taker_buy_base'] - df['taker_sell_base']
    df['cvd'] = df['delta_volume'].cumsum()
    df['cvd_imbalance'] = df['delta_volume'] / (df['volume'] + 1e-10)

    # Pillar 7: VWAP & Bands
    typical_price = (df['high'] + df['low'] + df['close']) / 3.0
    cum_pv = (typical_price * df['volume']).cumsum()
    cum_vol = df['volume'].cumsum()
    df['vwap'] = cum_pv / (cum_vol + 1e-10)
    dev = ((typical_price - df['vwap']) ** 2 * df['volume']).cumsum() / (cum_vol + 1e-10)
    df['vwap_std'] = np.sqrt(np.maximum(dev, 1e-10))
    df['vwap_zscore'] = (df['close'] - df['vwap']) / (df['vwap_std'] + 1e-10)

    # Pillar 8: RSI 14
    delta = df['close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-10)
    df['rsi14'] = 100.0 - (100.0 / (1.0 + rs))

    # Pillar 9: MACD Momentum
    ema12 = df['close'].ewm(span=12, adjust=False).mean()
    ema26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd_line'] = ema12 - ema26
    df['macd_signal'] = df['macd_line'].ewm(span=9, adjust=False).mean()
    df['macd_hist'] = df['macd_line'] - df['macd_signal']
    df['macd_accel'] = df['macd_hist'] - df['macd_hist'].shift(1)

    # Pillar 1 & 3: Structure & Liquidity Sweeps
    df['swing_high_20'] = df['high'].rolling(20).max()
    df['swing_low_20'] = df['low'].rolling(20).min()
    df['dist_to_swing_high'] = (df['swing_high_20'] - df['close']) / df['close']
    df['dist_to_swing_low'] = (df['close'] - df['swing_low_20']) / df['close']

    upper_wick = df['high'] - np.maximum(df['open'], df['close'])
    lower_wick = np.minimum(df['open'], df['close']) - df['low']
    body_size = (df['close'] - df['open']).abs() + 1e-6
    df['sweep_score'] = np.where((lower_wick > 2.0 * body_size) & (df['volume'] > df['volume'].rolling(10).mean() * 1.5), 1.0,
                        np.where((upper_wick > 2.0 * body_size) & (df['volume'] > df['volume'].rolling(10).mean() * 1.5), -1.0, 0.0))

    # Pillar 2: Fair Value Gap (FVG)
    df['bullish_fvg'] = np.where(df['low'] > df['high'].shift(2), 1.0, 0.0)
    df['bearish_fvg'] = np.where(df['high'] < df['low'].shift(2), -1.0, 0.0)
    df['fvg_bias'] = df['bullish_fvg'] + df['bearish_fvg']

    # Pillar 10: Fibonacci Golden Pocket (61.8% OTE)
    price_range = df['swing_high_20'] - df['swing_low_20'] + 1e-6
    fib_618 = df['swing_high_20'] - 0.618 * price_range
    df['fib_golden_proximity'] = 1.0 - np.clip(np.abs(df['close'] - fib_618) / price_range, 0.0, 1.0)

    df.dropna(inplace=True)
    return df

feature_df = compute_10_pillar_features(raw_df)
print(f"✅ 10-Pillar Features engineered: {feature_df.shape}")

### 🎯 Step 4: Asymmetric Expected Value Labeling (R:R >= 1:2.5)

In [ ]:
def generate_asymmetric_labels(df, forward_bars=16, tp_pct=0.025, sl_pct=0.010):
    closes = df['close'].values
    highs = df['high'].values
    lows = df['low'].values
    n = len(df)
    labels = np.zeros(n, dtype=int)

    for i in range(n - forward_bars):
        entry_p = closes[i]
        tp_price = entry_p * (1.0 + tp_pct)
        sl_price = entry_p * (1.0 - sl_pct)

        for j in range(i + 1, i + forward_bars + 1):
            if lows[j] <= sl_price:
                labels[i] = 0
                break
            if highs[j] >= tp_price:
                labels[i] = 1
                break

    df['target'] = labels
    return df

labeled_df = generate_asymmetric_labels(feature_df, forward_bars=16, tp_pct=0.025, sl_pct=0.010)
print(f"✅ Labeled Targets: {labeled_df['target'].sum()} positive signals out of {len(labeled_df)} ({labeled_df['target'].mean()*100:.2f}% base rate)")

### 🚀 Step 5: Train Models (HMM, Triple Ensemble, PatchTST, MoE)

In [ ]:
os.makedirs("models", exist_ok=True)

feature_cols = [
    'ema_alignment', 'atr_normalized', 'cvd_imbalance', 'vwap_zscore',
    'rsi14', 'macd_hist', 'macd_accel', 'dist_to_swing_high',
    'dist_to_swing_low', 'sweep_score', 'fvg_bias', 'fib_golden_proximity'
]

X = labeled_df[feature_cols].values
y = labeled_df['target'].values

split_idx = int(len(X) * 0.80)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, "models/brain_scaler.pkl")

# 1. HMM Market Regime
print("Training HMM Market Regime Classifier...")
from hmmlearn import hmm
hmm_model = hmm.GaussianHMM(n_components=3, covariance_type="diag", n_iter=100, random_state=42)
hmm_model.fit(X_train_scaled[:, :4])
joblib.dump(hmm_model, "models/brain_hmm_regime.pkl")

# 2. XGBoost
print("Training XGBoost Classifier...")
from xgboost import XGBClassifier
xgb_clf = XGBClassifier(n_estimators=250, learning_rate=0.03, max_depth=5, random_state=42)
xgb_clf.fit(X_train_scaled, y_train)
joblib.dump(xgb_clf, "models/brain_xgb.pkl")

# 3. CatBoost
print("Training CatBoost Classifier...")
from catboost import CatBoostClassifier
cb_clf = CatBoostClassifier(iterations=250, learning_rate=0.03, depth=5, verbose=0, random_seed=42)
cb_clf.fit(X_train_scaled, y_train)
joblib.dump(cb_clf, "models/brain_catboost.pkl")

# 4. LightGBM
print("Training LightGBM Classifier...")
from lightgbm import LGBMClassifier
lgb_clf = LGBMClassifier(n_estimators=250, learning_rate=0.03, max_depth=5, random_state=42, verbose=-1)
lgb_clf.fit(X_train_scaled, y_train)
joblib.dump(lgb_clf, "models/brain_lightgbm.pkl")

# 5. PatchTST Transformer
print("Training PatchTST Transformer...")
import tensorflow as tf
from tensorflow.keras import layers, models
inputs = layers.Input(shape=(X_train_scaled.shape[1], 1))
x = layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')(inputs)
attn = layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
x = layers.Add()([x, attn])
x = layers.LayerNormalization()(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
patch_model = models.Model(inputs=inputs, outputs=outputs)
patch_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
X_3d = np.expand_dims(X_train_scaled, axis=-1)
patch_model.fit(X_3d, y_train, epochs=12, batch_size=32, verbose=0)
patch_model.save("models/brain_patchtst.h5")

# 6. MoE Gating Router
print("Training MoE Gating Router...")
moe_router = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
moe_router.fit(X_train_scaled, y_train)
joblib.dump(moe_router, "models/brain_moe_router.pkl")

print("\n🎉 [SUCCESS] All 6 Super Smart Models Trained & Saved in 'models/' directory!")

### 🤗 Step 6: Push Trained Models to Hugging Face Model Hub

Enter your Hugging Face Access Token below (with `write` permission) to upload all weights directly to `hemsinath/apex-ai-brain-models`.

In [ ]:
from huggingface_hub import HfApi

# Set your HF Token here or via Colab Secret:
HF_TOKEN = "" # <-- Paste your Hugging Face Token here
REPO_ID = "hemsinath/apex-ai-brain-models"

if not HF_TOKEN:
    HF_TOKEN = os.getenv("HF_TOKEN", "")

if HF_TOKEN:
    api = HfApi()
    files = [f for f in os.listdir("models") if f.endswith(('.pkl', '.h5', '.json'))]
    print(f"Uploading {len(files)} model files to Hugging Face Hub ({REPO_ID})...")
    for f in files:
        path = os.path.join("models", f)
        print(f"  └─ Uploading {f}...")
        api.upload_file(
            path_or_fileobj=path,
            path_in_repo=f,
            repo_id=REPO_ID,
            token=HF_TOKEN
        )
    print(f"\n🎉 [ALL MODELS SYNCED] Visit: https://huggingface.co/{REPO_ID}")
    print("👉 On your VPS or local machine, simply run: python sync_local_models.py")
else:
    print("⚠️ Please provide HF_TOKEN to auto-upload to Hugging Face Hub.")